# Strategic Thinking
Strategic thinking provides a structured reflection mechanism for agents to reflect on progress before deciding the next action, assess gaps, and plan next steps in their research workflow.

In [ ]:
from typing import Annotated, Literal

from langchain.agents import AgentState, create_agent
from langchain.messages import ToolMessage
from langchain.tools import InjectedState, InjectedToolCallId, tool
from langgraph.types import Command
from loguru import logger
from pydantic import BaseModel

from chain_reaction.config import ModelName, get_chat_model
from chain_reaction.reducers import reduce_dict
from chain_reaction.utils import format_messages

## Think tool

In [ ]:
@tool
def think(reflection: str) -> str:
    """Pause to reflect on progress before deciding the next action.

    Use at natural checkpoints — after a search, a function, a subtask, or any
    step whose outcome should shape what you do next.

    When to use:
    - After a discrete step: What did I just learn or produce?
    - Before deciding next steps: Do I have what I need to move forward, or to finish?
    - When assessing gaps: What is still unknown, undecided, or incomplete?
    - When complexity rises: Am I on the right path, or should I change approach?
    - Before concluding: Can I deliver a complete, correct result now?

    When NOT to use:
    - The next action is already obvious.
    - The step was mechanical (formatting, a rename, a trivial edit).
    - You just reflected and nothing material has changed.

    Reflection should consider any of the following that apply:
    1. Current state - What concrete progress, findings, or output do I have?
    2. Gaps - What is still unknown, undecided, or incomplete?
    3. Quality - Is the work so far sound and sufficient?
    4. Next move - Continue, pivot, or finish?

    Keep it to 1-4 sentences. Go longer only when the decision is genuinely hard.

    Args:
        reflection (str): Your reflection on progress, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    logger.info("🧠 thinking...")
    return f"Reflection recorded: {reflection}"

## Planning tools

In [ ]:
type Status = Literal["pending", "in-progress", "completed", "not-needed"]


class Task(BaseModel):
    """Task item for tracking progress through complex workflows.

    Attributes:
        title (str): Task title.
        description (str): Detailed description of task.
        status (Status): Current status of task. Defaults to "pending".
    """

    title: str
    description: str
    status: Status = "pending"


class TaskStatusUpdate(BaseModel):
    """Status update for a single task.

    Attributes:
        task_idx (int): Index of task to update.
        status (Status): New status for the task.
    """

    task_idx: int
    status: Status


class DeepAgentState(AgentState):
    """Extended agent state that includes task tracking."""

    tasks: Annotated[dict[int, Task], reduce_dict]


@tool
def append_tasks(
    tasks: list[Task],
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Append new (pending) tasks for tracking progress through complex workflows.

    ## When to create tasks
    - Multi-step or non-trivial tasks requiring coordination or extended reasoning
    - When user provides multiple tasks or explicitly requests todo list
    - Avoid for single, trivial actions unless directed otherwise

    Args:
        tasks (list[Task]): Tasks to add.
        state (Annotated[DeepAgentState, InjectedState]): Injected agent state containing the current tasks.
        tool_call_id (str): Tool call identifier for message response.

    Returns:
        Command: Command to update agent state with new tasks.
    """
    current_tasks: dict[int, Task] = state.get("tasks", {})
    new_tasks = dict(enumerate(tasks, start=len(current_tasks)))
    logger.info("📝 Appending {num_tasks} to task list", num_tasks=len(new_tasks))
    return Command(
        update={
            "tasks": new_tasks,
            "messages": [ToolMessage(f"Added {len(new_tasks)} to the task list", tool_call_id=tool_call_id)],
        }
    )


@tool
def update_task_status(
    updates: list[TaskStatusUpdate],
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Update the status of one or more tasks.

    Use this when starting a task (pending -> in-progress) or when a task is completed (in-progress -> complete).

    Args:
        updates (list[TaskStatusUpdate]): Task index and new status for each update.
        state (Annotated[DeepAgentState, InjectedState]): Injected agent state containing the current tasks.
        tool_call_id (str): Tool call identifier for message response.

    Returns:
        Command: Command to update status of specified tasks.
    """
    logger.info("🔄 Updating status of {num_tasks} tasks", num_tasks=len(updates))

    current_tasks: dict[int, Task] = state.get("tasks", {})
    updated_tasks = {
        u.task_idx: current_tasks[u.task_idx].model_copy(update={"status": u.status})
        for u in updates
        if u.task_idx in current_tasks
    }

    requested = {u.task_idx for u in updates}
    if not_updated := requested - updated_tasks.keys():
        logger.info("Failed to update indexes {not_updated}", not_updated=len(not_updated))
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        f"Failed to update indexes {not_updated}. Available indexes: {list(current_tasks.keys())}",
                        tool_call_id=tool_call_id,
                    )
                ],
            }
        )

    return Command(
        update={
            "tasks": updated_tasks,
            "messages": [ToolMessage(f"Updated status of {len(updated_tasks)} tasks", tool_call_id=tool_call_id)],
        }
    )


@tool
def view_tasks(
    state: Annotated[DeepAgentState, InjectedState],
) -> str:
    """View the task index, task title and status of each task in the current task list.

    Each item in list is formatted as {task_idx}. {task_title}: {task_status}

    Args:
        state (Annotated[DeepAgentState, InjectedState]): Injected agent state containing the current tasks.

    Returns:
        str: Formatted string representation of the current task list.
    """
    current_tasks: dict[int, Task] | None = state.get("tasks", {})
    logger.info("👀 Viewing {num_tasks} tasks from task list", num_tasks=len(current_tasks))
    if not current_tasks:
        return "No tasks"
    return "\n".join(f"{task_idx}. {task.title}: {task.status}" for task_idx, task in current_tasks.items())


@tool
def read_task(
    task_idx: int,
    state: Annotated[DeepAgentState, InjectedState],
) -> str:
    """Read full task title, status, and description from task list.

    Always read the full task description before starting any task. Re-read task description
    when finishing a task to make sure it was completed in full.

    Args:
        task_idx (int): Task index in list.
        state (Annotated[DeepAgentState, InjectedState]): Injected agent state containing the current tasks.

    Returns:
        str: Task title, status, and description from task list, or error message if task doesn't exist.
    """
    logger.info("📖 Reading task {task_idx}", task_idx=task_idx)
    current_tasks: dict[int, Task] | None = state.get("tasks", {})

    if task_idx not in current_tasks:
        return f"Task index {task_idx} does not exist. Available indexes: {list(current_tasks.keys())}"

    task = current_tasks[task_idx]
    return f"{task.title}: {task.status}\n{task.description}"


task_management_tools = [append_tasks, update_task_status, view_tasks, read_task]
TASK_MANAGEMENT_INSTRUCTIONS = f"""Based upon the user's request:
1. Use the {append_tasks.func.__name__} tool to create a task list at the start of a user request.
2. Use {view_tasks.func.__name__} to view the status of the current task list and reflect on what you've done and the TODO.
3. When you start working or after you accomplish any task, use the {update_task_status.func.__name__} to update status.
If a task becomes irrelevant mark it as 'not-needed' and move on to the next item.
4. Continue this process until you have completed all tasks.

## Best Practices
- New tasks should always be created as 'pending'
- Only one 'in-progress' task at a time
- Mark 'completed' immediately when task is fully done
- Prune irrelevant tasks by marking as 'not-needed'

## Progress Updates
- Call {update_task_status.func.__name__} function again to change task status
- Reflect real-time progress; don't batch completions
- If blocked, keep blocked task 'in-progress' and add new task describing blocker

## Structure
- Maintain one list containing multiple task objects (title, description, status)
- Use clear, actionable task descriptions
- Status must be: 'pending', 'in-progress', 'completed', or 'not-needed'

IMPORTANT: Always create a plan of tasks for ANY user request.
"""  # noqa: E501

## Calculator tool

In [ ]:
type Numeric = int | float
type Operation = Literal["add", "subtract", "multiply", "divide"]


@tool
def calculator(  # noqa: C901
    operation: Operation,
    a: Numeric,
    b: Numeric,
) -> Numeric | str:
    """Apply an arithmetic operation to two numbers and return result.

    Args:
        operation (Operation): Operation to apply.
        a (Numeric): First number
        b (Numeric): Second number

    Returns:
        Numeric | str: Result of arithmetic operation applied to the two input numbers, or error message.
    """
    logger.info("🔢 calculator: {operation}({a}, {b})", operation=operation, a=a, b=b)
    match operation:
        case "add":
            return a + b
        case "subtract":
            return a - b
        case "multiply":
            return a * b
        case "divide":
            if b == 0:
                return "error: Cannot divide by 0."
            return a / b
        case _:
            return f"error: {operation} not supported."

## Agent

In [ ]:
agent = create_agent(
    model=get_chat_model(model_name=ModelName.CLAUDE_HAIKU),
    tools=[calculator, think],  # , *task_management_tools],
    system_prompt="""
    You are a helpful question answering assistant. Use your tools
    to answer the user's question as accurately as possible.

    Use the calculator for math, don't guess.
    """,
    state_schema=DeepAgentState,
)

In [ ]:
question = """
Three friends — Alex, Ben, and Chloe — perform the following six transactions, in exact order:

Alex gives Ben an amount equal to 20% of Alex's current money
Ben gives Chloe an amount equal to 25% of Ben's current money
Chloe gives Alex an amount equal to 40% of Chloe's current money
Alex gives Ben an amount equal to 50% of Alex's current money
Ben gives Chloe an amount equal to 25% of Ben's current money
Chloe gives Alex an amount equal to 50% of Chloe's current money
After all six transactions:

Alex has $99
Ben has $99
Chloe has $42
Each person started with a whole-dollar amount. How much did each start with?
"""

# Expected correct answer
# Alex: $100, Ben: $80, Chloe: $60 (total $240)

response = agent.invoke(input={"messages": [("user", question)]})
format_messages(response.get("messages", []))